# 2주차 5교시 · 숫자 열과 문자 열을 처리할 전처리 순서 만들기

**학생용 지적실험노트**

오늘의 질문: **숫자와 문자가 섞인 13개 열을 한 번에 같은 방식으로 처리하려면 무엇을 준비해야 할까요?**

오늘의 완성 결과: 숫자 9개 열과 문자 4개 열의 처리 순서를 preprocessor 객체 하나에 담습니다.

## 시작하기 전에 폴더를 확인하세요

- `activity_batches`는 강사가 ZIP 안에 미리 넣어 준 폴더입니다. 학생이 새로 만들지 않습니다.
- ZIP을 먼저 압축 해제하고, 압축을 풀어서 생긴 최상위 폴더 전체를 VS Code에서 엽니다.
- 이 노트북이나 CSV 파일을 다른 폴더로 따로 옮기지 않습니다.
- 자세한 순서는 `week2/00_학생용_먼저읽기_압축해제와VSCode열기.md`를 확인합니다.

각 Q에서 ① 실행 전 예상 ② ___ 채우기 ③ 실행 ④ 값 하나 바꾸기 ⑤ 결과 이유 한 문장을 지킵니다. 오류 메시지에 ___가 보이면 고장이 아니라 아직 코드 빈칸이 남았다는 뜻입니다.


In [ ]:
from pathlib import Path
import pandas as pd

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
print("데이터 폴더:", DATA_DIR)
batch_files = sorted((DATA_DIR / "activity_batches").glob("activity_batch_*.csv"))
frames = []
for file_path in batch_files:
    one_batch = pd.read_csv(file_path)
    one_batch["source_file"] = file_path.name
    frames.append(one_batch)
activity = pd.concat(frames, ignore_index=True)

activity["activity_dt"] = pd.to_datetime(activity["activity_date"], errors="coerce")
activity["stay_time_min_num"] = pd.to_numeric(activity["stay_time_min"], errors="coerce")
activity["active_dt"] = activity["activity_dt"].where(activity["logged_in"])

member_activity = activity.groupby("member_id").agg(
    active_days=("logged_in", "sum"),
    total_app_opens=("app_opens", "sum"),
    total_stay_time_min=("stay_time_min_num", "sum"),
    total_profile_views=("profile_views", "sum"),
    total_recommendations_viewed=("recommendations_viewed", "sum"),
    total_messages_sent=("messages_sent", "sum"),
    avg_reply_delay_sec=("avg_reply_delay_sec", "mean"),
    last_active_date=("active_dt", "max"),
).reset_index()
reference_date = pd.Timestamp("2026-07-01")
member_activity["days_since_last_activity"] = (
    reference_date - member_activity["last_active_date"]
).dt.days

members = pd.read_csv(DATA_DIR / "members.csv")
member_columns = [
    "member_id", "gender", "age", "residence_region",
    "membership_tier", "signup_channel", "meeting_scheduled"
]
model_table = members[member_columns].merge(
    member_activity, on="member_id", how="left", validate="one_to_one"
)
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


## Q1. 계산할 숫자 열과 종류를 나타내는 문자 열을 나눕니다

**먼저 예상하기**

- 왜 13개 열을 숫자와 문자 목록으로 따로 적어 둘까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 모든 열에 같은 처리를 적용하지 않고 자료의 성격에 맞는 처리 순서를 정합니다.

**현재 자료구조** numeric_features는 문자열 9개 list, categorical_features는 문자열 4개 list입니다.

**코드 읽기 도움** 이 list는 데이터 열을 복사한 것이 아니라, 사용할 열 이름 문자열을 순서대로 적은 설계도입니다. Q4의 model_table.columns를 보고 실제 철자를 그대로 적습니다.

**빈칸 채우는 순서** model_table의 16개 열 중 member_id(식별자), meeting_scheduled(정답), last_active_date(원본 날짜)는 이번 입력 목록에서 제외합니다. 남은 열을 숫자로 계산할 열 9개와 문자 범주 열 4개로 나눕니다. 마지막 print의 빈칸에는 목록의 길이를 세는 함수를 넣습니다.

**정상 결과** 9 4

**오류가 나면** KeyError. model_table.columns와 열 이름의 철자를 비교합니다.


In [ ]:
numeric_features = ["___", "___", "___", "___", "___", "___", "___", "___", "___"]
categorical_features = ["___", "___", "___", "___"]
print(___(numeric_features), ___(categorical_features))

## Q2. 13개 입력 열 X와 일정 등록 여부 y를 따로 둡니다

**먼저 예상하기**

- meeting_scheduled가 X에 함께 들어가면 왜 문제가 될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 입력 정보와 맞힐 값을 섞지 않아 답안 유출을 막습니다.

**현재 자료구조** X는 1,000행×13열 DataFrame, y는 True/False 1,000개 Series입니다.

**코드 읽기 도움** X는 여러 열을 가진 표이고 y는 정답 한 열입니다. 여기서는 전처리 자동화를 연습하기 위한 y입니다.

**정상 결과** (1000, 13) (1000,)

**오류가 나면** meeting_scheduled가 X.columns에 남음. X.columns를 출력하고 정답 열을 선택 목록에서 뺍니다.


In [ ]:
X = model_table[___ + ___]
y = model_table[___]
print(X.shape, y.shape)

## Q3. 회원 1,000명을 train 800명과 test 200명으로 함께 나눕니다

**먼저 예상하기**

- X와 y를 따로 나누면 왜 같은 회원끼리 어긋날 수 있을까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 전처리 기준은 train에서만 배우고 test는 같은 기준을 확인하는 데 사용합니다.

**현재 자료구조** X_train 800×13, X_test 200×13, y_train/y_test는 각각 800개와 200개 Series입니다.

**코드 읽기 도움** stratify=y는 True/False 비율이 train과 test에서 비슷하게 유지되도록 나눕니다.

**정상 결과** (800, 13) (200, 13)

**오류가 나면** 행 수가 다르거나 비율이 크게 달라짐. test_size와 stratify=y를 확인합니다.


In [ ]:
___, ___, ___, ___ = ___(
    ___, ___,
    ___=0.2,
    ___=42,
    ___=y
)
print(X_train.shape, X_test.shape)

## Q4. 숫자 열은 빈값을 채운 뒤 크기 기준을 맞추도록 설계합니다

**먼저 예상하기**

- 연봉과 활동일 수처럼 단위가 다른 숫자를 같은 계산에 넣기 전에 무엇이 필요할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 결측 때문에 계산이 멈추지 않게 하고, 단위가 큰 열이 계산을 지배하지 않게 합니다.

**현재 자료구조** numeric_pipe는 두 단계 이름과 처리 객체를 순서대로 가진 Pipeline입니다.

**코드 읽기 도움** Pipeline 안의 tuple은 ('단계 이름', 실제 처리 객체) 한 쌍입니다.

**정상 결과** imputer 다음 scaler 두 단계

**오류가 나면** NameError 또는 단계 순서가 반대. Pipeline list의 위에서 아래 순서를 확인합니다.


In [ ]:
numeric_pipe = ___([
    ("imputer", ___(strategy=___)),
    ("scaler", ___()),
])
print(numeric_pipe.steps)

## Q5. 문자 열을 0·1 열로 바꾸고 숫자 처리와 한데 묶습니다

**먼저 예상하기**

- 지역이나 회원권처럼 글자로 된 값은 어떻게 계산 가능한 열로 바꿀까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 처음 보는 범주가 들어와도 멈추지 않고 숫자·문자 처리를 한 번에 재현합니다.

**현재 자료구조** categorical_pipe는 두 단계 Pipeline, preprocessor는 두 Pipeline과 열 목록을 가진 ColumnTransformer입니다.

**코드 읽기 도움** ColumnTransformer는 숫자 열 list를 numeric_pipe로, 문자 열 list를 categorical_pipe로 보낸 뒤 결과를 옆으로 합칩니다.

**정상 결과** 숫자 처리(num)와 문자 처리(cat)가 함께 표시된 ColumnTransformer

**오류가 나면** OneHotEncoder sparse 관련 오류. sparse_output=False와 transform='pandas'를 함께 확인합니다.


In [ ]:
categorical_pipe = ___([
    ("imputer", ___(___="most_frequent")),
    ("onehot", ___(___="ignore", ___=False)),
])
preprocessor = ___([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
], verbose_feature_names_out=False).set_output(___="pandas")
print(preprocessor)

## 마무리 확인

- [ ] 오늘 결과가 **숫자 9개 열과 문자 4개 열의 처리 순서를 preprocessor 객체 하나에 담습니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
